*MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA*

**Curso: TC4034.10- Análisis de grandes volúmenes de datos**

Tecnológico de Monterrey

Dr. Iván Olmos Pineda

Estudiante: Gloria María Campos García 						A01422345

**Actividad 3** |  Aprendizaje supervisado y no supervisado

### 1. Introducción

El aprendizaje automático en PySpark se divide principalmente en dos categorías, las cuales son el aprendizaje supervisado y el aprendizaje no supervisado.

El aprendizaje supervisado es el proceso de alimentar un modelo con datos de entrenamiento que incluyen etiquetas para que el modelo aprenda a resolver una tarea específica, como clasificar objetos o predecir valores continuos (Polak, 2023).
Los algoritmos representativos en PySpark incluyen:
* Logistic Regression
* Decision Tree Classifier
* Random Forest Classifier
* GBTClassifier
* Multilayer Perceptron

Por otro lado, el aprendizaje no supervisado se utiliza para encontrar estructuras o agrupaciones ocultas en datos que no tienen etiquetas previas, siendo el agrupamiento (clustering) una de sus  principales aplicaciones  (Polak, 2023).
Algunod de los algoritmos disponibles en PySpark son:
* K-means
* Gaussian Mixture (GMM)
* Latent Dirichlet Allocation (LDA)
* Power Iteration Clustering (PIC)

### 2. Selección de los datos (Construcción de la Muestra M)
#### Nombre y origen del dataset
Nombre: NYC Yellow Taxi Trip Data
Origen: Los datos son recopilados por los proveedores autorizados de tecnología TPEP Creative Mobile Technologies y VeriFone Inc. y publicados oficialmente por la NYC Taxi & Limousine Commission (TLC), la entidad reguladora del transporte de pasajeros en la ciudad de Nueva York.
El dataset utilizado es el que se encuentra en [kaggle](https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data).

#### Particionamiento de la base de datos (D)
Con el propósito de elaborar una muestra (M) representativa de la población de usuarios de taxis amarillos en la ciudad de Nueva York, se diseñó un proceso de segmentación de la base de datos (D) empleando variables asociadas con los patrones de movilidad urbana y el transporte.
El objetivo de este particionamiento radica en dividir la base de datos inicial en subconjuntos homogéneos que reflejen las diversas pautas de movilidad.
* Particionamiento: Segmentar por time_period (Madrugada, Mañana, Tarde, Noche) y distance_category (Corta, Media, Larga).
* Muestreo Aleatorio Simple (MAS): Recuperar instancias de cada una de las 6 combinaciones de particiones de forma proporcional a su frecuencia relativa.

#### 2.1. Configuración del entorno

In [ ]:
#Librerias
import findspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, ClusteringEvaluator

In [ ]:
findspark.init(r"C:\\Users\\gloca\\OneDrive\\Documents\\spark-4.1.1-bin-hadoop3\\spark-4.1.1-bin-hadoop3")

In [ ]:
#Configuración de la sesión
spark = SparkSession.builder \
    .appName("Recuperacion_Particiones_Taxi_ML") \
    .getOrCreate()

#### 2.2. Cargar los datos originales (Población D)

In [ ]:
def read_and_clean(path):
  # Cargamos con inferSchema=True para asegurar tipos correctos en el filtrado
    df = spark.read.csv(path, header=True, sep=",", inferSchema=True)
    return df

path_base = "archive\\"
df1 = read_and_clean(path_base + "yellow_tripdata_2015-01.csv")
df2 = read_and_clean(path_base + "yellow_tripdata_2016-01.csv")
df3 = read_and_clean(path_base + "yellow_tripdata_2016-02.csv")
df4 = read_and_clean(path_base + "yellow_tripdata_2016-03.csv")

In [ ]:
# Unión de la base de datos completa D (Población total de aprox 47.2M registros)
df_D = df1.unionByName(df2).unionByName(df3).unionByName(df4)
print(f"Base de datos D cargada. Total de registros: {df_D.count()}")

Base de datos D cargada. Total de registros: 47248845


#### 2.3. Reglas de las submuestras

In [ ]:
df_reglas = df_D.withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("time_period",
        F.when((F.col("pickup_hour") >= 0) & (F.col("pickup_hour") < 6), "Madrugada")
        .when((F.col("pickup_hour") >= 6) & (F.col("pickup_hour") < 12), "Manana")
        .when((F.col("pickup_hour") >= 12) & (F.col("pickup_hour") < 18), "Tarde")
        .otherwise("Noche")) \
    .withColumn("distance_category",
        F.when(F.col("trip_distance") < 2, "Corta")
        .when((F.col("trip_distance") >= 2) & (F.col("trip_distance") < 10), "Media")
        .otherwise("Larga"))

#### 2.4. Construcción de la Muestra M (Recuperación proporcional por partición)
Se considerará recuperar un 1% de cada partición para evitar altos tiempos de procesamiento

In [ ]:
periodos = ["Madrugada", "Manana", "Tarde", "Noche"]
distancias = ["Corta", "Media", "Larga"]
M = None

for p in periodos:
    for d in distancias:
        particion = df_reglas.filter((F.col("time_period") == p) & (F.col("distance_category") == d))
        # MAS dentro de cada partición
        sub_muestra = particion.sample(withReplacement=False, fraction=0.01, seed=42)
        M = sub_muestra if M is None else M.union(sub_muestra)

print(f"Muestra M construida con {M.count()} registros.")

Muestra M construida con 476488 registros.


### 3. Preparación de los datos (Muestra M Pre-procesada)

Se deben corregir inconsistencias detectadas en la base de datos original, como fechas fuera de rango, coordenadas en cero y valores monetarios negativos.
* Limpieza: Eliminar o imputar valores nulos usando dropna o fillna
* Filtrado: Remover registros con fare_amount negativo o coordenadas geográficas inválidas
* Transformación: Convertir columnas de tipo string a numérico (Double/Integer) según corresponda
* Vectorización: Usar VectorAssembler para consolidar las características en una sola columna llamada features

#### 3.1. Limpieza y corrección de inconsistencias

In [ ]:
M_preprocesada = M.dropna() \
    .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0)) \
    .filter((F.col("pickup_longitude") != 0) & (F.col("pickup_latitude") != 0))

#### 3.2. Transformación de tipos y Vectorización

In [ ]:
# Indexamos payment_type como nuestra variable objetivo numérica
indexer = StringIndexer(inputCol="payment_type", outputCol="label")
M_indexada = indexer.fit(M_preprocesada).transform(M_preprocesada)

# Ensamblador para las columnas de movilidad
required_features = ["passenger_count", "trip_distance", "fare_amount", "pickup_longitude", "pickup_latitude"]
assembler = VectorAssembler(inputCols=required_features, outputCol="features")
M_final = assembler.transform(M_indexada)

### 4. Preparación del conjunto de entrenamiento y prueba
Para minimizar el sesgo, se utilizará la técnica de randomSplit con una división de 70% para entrenamiento y 30% para prueba. Esta división se realiza para garantizar que el modelo tenga suficientes ejemplos para aprender (70%) mientras se reserva una porción robusta (30%) para validar su capacidad de generalización sobre datos no vistos.


In [ ]:
# División aleatoria con semilla para reproducibilidad
train_df, test_df = M_final.randomSplit([0.7, 0.3], seed=42)
print(f"Set Entrenamiento: {train_df.count()} | Set Prueba: {test_df.count()}")

Set Entrenamiento: 325954 | Set Prueba: 140188


### 5. Construcción de modelos de aprendizaje supervisado y no supervisado


#### 5.1. Aprendizaje Supervisado: Random Forest Classifier

Se predice el tipo de pago (label) usando las características del viaje, considerando la Exactitud (Accuracy) como métrica de calidad.

In [ ]:
# Entrenamiento de Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10)
model_rf = rf.fit(train_df)

# Predicciones y Evaluación
predictions_rf = model_rf.transform(test_df)
evaluator_rf = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
accuracy = evaluator_rf.evaluate(predictions_rf)
print(f"Calidad Supervisada (Accuracy): {accuracy:.4f}")

Calidad Supervisada (Accuracy): 0.6598


#### 5.2. Aprendizaje No Supervisado: K-means Clustering

Se agrupan los viajes por patrones de movilidad. Utilizando el Silhouette Score para medir la calidad del agrupamiento, buscando una alta cohesión interna.

In [ ]:
# Entrenamiento de K-means
kmeans = KMeans(k=3, seed=1, featuresCol="features")
model_km = kmeans.fit(M_final)

# Evaluación del Clúster
predictions_km = model_km.transform(M_final)
evaluator_km = ClusteringEvaluator()
silhouette = evaluator_km.evaluate(predictions_km)
print(f"Calidad No Supervisada (Silhouette Score): {silhouette:.4f}")

Calidad No Supervisada (Silhouette Score): 1.0000


#### Conclusión
El algoritmo Random Forest Classifier alcanzó una exactitud (Accuracy) de 0.6598. Este resultado indica que el modelo es capaz de predecir correctamente la variable objetivo (como el payment_type o método de pago) en aproximadamente el 66% de los casos. En el contexto de los taxis de NYC, donde existen múltiples categorías de pago e influencias externas, esta precisión sugiere que, aunque el modelo captura patrones significativos en las características de movilidad (como distancia y tarifa), todavía existe una variabilidad considerable en los datos que dificulta una clasificación perfecta.

Por otro lado, el algoritmo K-means arrojó un Silhouette Score de 1.0000. Desde una perspectiva teórica, este es el valor máximo posible, lo que indica una separación perfecta entre los clústeres y una cohesión total dentro de cada grupo. En la práctica, un puntaje de 1.0 suele ser poco común en datos reales de gran volumen y sugiere que los grupos identificados (por ejemplo, viajes cortos económicos vs. viajes largos costosos) están extremadamente diferenciados entre sí sin ningún traslape en el espacio de características seleccionado.Este resultado valida que el particionamiento previo de la base de datos D en subconjuntos homogéneos facilitó la identificación de patrones de movilidad muy claros.

### Bibliografía

* DecisionForest. (s.f.). Complete Machine Learning Project with PySpark MLlib Tutorial [Archivo de vídeo]. YouTube.
* Equipo 31. (2026). Proyecto | Base de Datos de Big Data. Instituto Tecnológico y de Estudios Superiores de Monterrey.
* Equipo 31. (2026). Proyecto | Lectura, Escritura, Archivos de Big Data PySpark. Instituto Tecnológico y de Estudios Superiores de Monterrey.
* Polak, A. (2023). Scaling Machine Learning with Spark. O'Reilly Media, Inc.
* Stats Wire. (s.f.). PySpark Tutorial 36: PySpark K Means Clustering [Archivo de vídeo]. YouTube.
* Stats Wire. (s.f.). PySpark Tutorial 37: PySpark VectorAssembler [Archivo de vídeo]. YouTube.